# Quantra — Train on Colab

One universal **PPO** policy built to **repeatedly pass FTMO-style challenges** (2.5% daily target / 4% trailing wall) — not to maximize PnL.

An episode is **N trading days on ONE continuous account** (C10): the balance carries forward day to day, a day that hits the wall is **locked out for the rest of that day** and resets at midnight (it does **not** end the episode), and the episode ends only when the N days are exhausted or the account is blown. A **large end-of-day penalty** (C11) fires whenever the bot misses that day's target, so it learns **consistency**, not just survival.

> ⚠️ **Pick a CPU runtime** (Runtime → Change runtime type → CPU, or High-RAM). The locked net is a tiny **3×256 MLP (~207 inputs)** — the bottleneck is CPU env-stepping, not the GPU. The optimizer below will *prove* whether a GPU is worth it (it usually isn't), so you don't waste paid GPU hours.

## 1. Clone the repo

In [ ]:
# The latest code lives on this BRANCH (change to "main" once merged). Without -b, Colab would clone
# the default branch and miss the new modules (barbershop_runner, repo_graph, the dashboard screens).
!git clone -b claude/focused-faraday-if1ue7 --single-branch https://github.com/monty313/RL-model-trading-bot-ppo-mlp_Claude-.git quantra_repo
%cd quantra_repo
!git rev-parse --abbrev-ref HEAD

## 2. Install dependencies
Colab already ships torch, numpy, pandas, scikit-learn, matplotlib. We only add the extras.

In [ ]:
!pip install -q gdown pyarrow psutil optuna seaborn statsmodels gymnasium tqdm
# nvidia-ml-py is only needed if you chose a GPU runtime:
import torch
if torch.cuda.is_available():
    !pip install -q nvidia-ml-py

## 3. Mount Google Drive (price data)
Your 4 symbols (MT5 1m, ~5yr) live in the `rl-trading-data` folder. Mounting lets the data loader read them without re-downloading 500 MB each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Expected path: /content/drive/MyDrive/rl-trading-data/<SYMBOL>_M1_*.csv
!ls -la "/content/drive/MyDrive/rl-trading-data" 2>/dev/null || echo "Adjust the folder path if your Drive layout differs; gdown-by-ID fallback is built into the loader."

## 4. Race the hardware (CPU vs GPU) and size to ~80%
This is the money-saver: it benchmarks both devices on the real four-head workload, picks the faster one (preferring CPU on a near-tie), scales parallel envs to ~80% utilisation, and reports what it measured.

In [ ]:
from quantra.runtime import RuntimeConfig, plan, print_report, UtilizationMonitor, ensure_dirs

ensure_dirs()
cfg = RuntimeConfig()
with UtilizationMonitor(interval=0.25) as mon:
    hw = plan(state_dim=cfg.nominal_state_dim, hw=cfg.hardware)
util = mon.stop()

print_report(hw)
print('Utilisation during race:', util.render())

## 5. HYPERPARAMETERS — all visible + editable BEFORE any run
Every knob the training run uses is surfaced here. **🔴 = LOCKED** (visible but read-only; changing needs Monty's sign-off). Edit the values, then build the env/trainer from the objects this cell constructs — nothing downstream is hardcoded.

In [ ]:
# ── HYPERPARAMETERS (edit here; gamma/lambda are 🔴 LOCKED — visible only) ──
from quantra.runtime import config as qcfg
from quantra.learning_system.trainer.gae import GAMMA, LAMBDA            # 🔴 0.997 / 0.97
from quantra.learning_system.trainer.trainer import TrainConfig
from quantra.learning_system.trainer.scheduler import AggressionRanges

# --- Episode length (C10: N trading days on ONE continuous account) ----------
TRAINING_DAYS = qcfg.TRAINING_DAYS     # default 180 (~6 months) for a full training run
EVAL_DAYS     = qcfg.EVAL_DAYS         # default 8 for a Barbershop/eval window
EPISODE_DAYS  = TRAINING_DAYS          # what THIS run uses (swap to EVAL_DAYS for a short eval)

# --- Challenge numbers (what "passing" means; → make_challenge) --------------
DAILY_TARGET_PCT   = 2.5               # daily profit goal (% of that day's OPENING balance)
DAILY_RISK_PCT     = 4.0               # daily trailing-wall risk
FTMO_MODE          = True              # ON: 2-phase auto-flat at target. OFF: target is the aim
STOP_FOR_DAY       = False             # OFF-mode: bank + lock out the day at target (else run on)
FAILED_DAY_PENALTY = 5.0               # C11 weight — LARGE end-of-day penalty, ∝ shortfall from target
PERMANENT_DD_PCT   = 10.0              # -10% max wall — OBSERVATION only (C12), NOT enforced

# --- Reward weights (C16: plain-English, operator-tunable; math/structure unchanged) ---
# Layer 0 net-PnL is the dominant outcome base E8 protects (keep net_pnl_weight = 1.0). The four
# shaping weights are small "whispers". daily_progress_weight is the consistency driver (matters most).
NET_PNL_WEIGHT         = 1.0           # 🔴 keep 1.0 (E8 Layer-0 dominance)
STEP_PNL_WEIGHT        = 1e-4          # small per-bar in-position bonus
DAILY_PROGRESS_WEIGHT  = 1e-3          # the consistency driver (raised from 1e-4)
DRAWDOWN_PAIN_WEIGHT   = 5e-4          # pain-zone penalty near the wall
DRAWDOWN_PAIN_STEEPNESS= 4.0           # exponential steepness of the pain ramp
TRADE_QUALITY_WEIGHT   = 5e-5          # close-quality / target-progress whisper

# --- Enforcement / safety toggles -------------------------------------------
TRAINING_WHEELS = qcfg.TRAINING_WHEELS  # counter-trend OPEN blocks (True = on)
TRAINING_PHASE  = qcfg.TRAINING_PHASE   # PHASE_FREE (signals observation-only) / PHASE_CONSTRAINED

# --- TEMPORARY experiment toggle (2026-06-20): reversible CCI-regime open-gate (OFF in the repo by
#     default). When True, a NEW open is allowed ONLY in a clean CCI regime — CCI30 AND CCI100 both
#     ABOVE their (existing period-2/shift-4) SMA on BOTH 1m AND 4H -> longs only; both BELOW ->
#     shorts only; mixed -> no new opens. Set False to remove the experiment entirely (the locked core
#     behaves identically either way). Injected into OVERRIDES below so it NAMES + reproduces the policy
#     and the eval/gauge enforce it via barbershop_runner.build_env. COUPLING ->
#     runtime/config.CCI_REGIME_GATE + env/trading_env.py direction_mask().
CCI_REGIME_GATE = True
# Per-trade RISK CAP (sizing) — RULE 3 overshoot fix: the 4%-wall breach already stops the day tick by
# tick, but an OVERSIZED position can blow THROUGH the wall in one bar (the -8/-10% days). Capping per-
# trade risk so 5 slots x this stays well under 4% keeps a single bar from overshooting. 0.01 = old 1%.
RISK_PER_TRADE = 0.005   # fraction of account risked per trade (5 slots x 0.5% = 2.5% max at risk < 4%)
# --- Exit overhaul [operator 2026-06-20]: cut losers small, let winners develop, pass the day fast ---
HARD_STOP_FRAC   = 0.005  # HARD per-trade stop = 0.5% of the INITIAL balance, checked every bar (0.0=off)
PROFIT_HOLD_WEIGHT   = 1e-3   # SMALL reward per profitable close held >= PROFIT_HOLD_MIN_BARS (folds into L4)
PROFIT_HOLD_MIN_BARS = 5      # a winner must be held this many minutes/bars to earn the profit-hold bonus
FAST_PASS_BONUS  = 5.0    # BIG once/day bonus for hitting +target AND being FLAT within FAST_PASS_HOURS
FAST_PASS_HOURS  = 12.0   # the window (from the day's open) the banked pass must land in to earn it

# --- PPO discount (🔴 LOCKED — shown for visibility, do not change) ----------
GAMMA_LOCKED  = GAMMA                  # 🔴 0.997  (the "math of patience")
LAMBDA_LOCKED = LAMBDA                 # 🔴 0.97

# --- Aggression dial RANGES (the missed-opportunity scheduler picks within these) ---
ENTROPY_RANGE = (0.03, 0.08)
CLIP_RANGE    = (0.25, 0.35)
LR_RANGE      = (5e-4, 1e-3)
EPOCHS_RANGE  = (10, 15)
# Exploration FLOOR (operator fix 2026-06-20): masks/guardrails hold the bot FLAT, which dilutes the
# G8 miss-rate toward 0 and makes the scheduler decay aggression (entropy/LR/epochs) to ~0 before a
# fresh policy learns. This floor keeps exploration of the still-legal options alive. 0.0 = locked
# behaviour; 0.3-0.4 keeps a healthy dial. COUPLING -> trainer/scheduler.AggressionScheduler(min_aggression=).
MIN_AGGRESSION = 0.35   # set 0.0 to reproduce the locked schedule exactly

# --- Rollout / optimization --------------------------------------------------
ROLLOUT_SIZE   = 512
MINIBATCH_SIZE = 64
VALUE_COEF     = 0.5

# Build the REAL config objects the env + Trainer consume (nothing hardcoded downstream).
CHALLENGE  = qcfg.make_challenge(daily_target_pct=DAILY_TARGET_PCT, daily_risk_pct=DAILY_RISK_PCT,
                                 ftmo_mode=FTMO_MODE, stop_for_day=STOP_FOR_DAY,
                                 permanent_dd_pct=PERMANENT_DD_PCT,
                                 failed_day_penalty=FAILED_DAY_PENALTY)
REWARD_CFG = qcfg.RewardConfig(net_pnl_weight=NET_PNL_WEIGHT, step_pnl_weight=STEP_PNL_WEIGHT,
                               daily_progress_weight=DAILY_PROGRESS_WEIGHT,
                               drawdown_pain_weight=DRAWDOWN_PAIN_WEIGHT,
                               drawdown_pain_steepness=DRAWDOWN_PAIN_STEEPNESS,
                               trade_quality_weight=TRADE_QUALITY_WEIGHT,
                               failed_day_penalty=FAILED_DAY_PENALTY,
                               profit_hold_weight=PROFIT_HOLD_WEIGHT, profit_hold_min_bars=PROFIT_HOLD_MIN_BARS,
                               fast_pass_bonus=FAST_PASS_BONUS, fast_pass_hours=FAST_PASS_HOURS)
TRAIN_CFG  = TrainConfig(rollout_size=ROLLOUT_SIZE, minibatch=MINIBATCH_SIZE, value_coef=VALUE_COEF)
AGG_RANGES = AggressionRanges(entropy=ENTROPY_RANGE, clip=CLIP_RANGE, lr=LR_RANGE, epochs=EPOCHS_RANGE)
RISK_CFG   = qcfg.RiskConfig(max_per_trade_risk_frac=RISK_PER_TRADE, hard_stop_frac=HARD_STOP_FRAC)  # sizing cap + hard stop
# (the env is built as TradingEnv(..., challenge=CHALLENGE, reward_cfg=REWARD_CFG, episode_days=EPISODE_DAYS))

# --- C19: OVERRIDES = ONLY the knobs that differ from baseline -> auto-names + reproduces the policy ---
# build_overrides_dict diffs the live config objects above against the dataclass defaults. The result
# NAMES the policy (auto_name) and is saved verbatim in its manifest by the Barbershop loop (Cell 6),
# so the Leaderboard can tell policies apart and every run is reproducible. Edit knobs ABOVE, not here.
from quantra.learning_system.policy_registry import auto_name
OVERRIDES = qcfg.build_overrides_dict(challenge=CHALLENGE, reward=REWARD_CFG, train=TRAIN_CFG,
                                      training_phase=TRAINING_PHASE, training_wheels=TRAINING_WHEELS)
if CCI_REGIME_GATE:
    OVERRIDES["cci_regime_gate"] = True           # TEMPORARY gate -> names the policy + reproduces the env
OVERRIDES["max_per_trade_risk_frac"] = RISK_PER_TRADE                 # RULE 3: same sizing cap in train + eval
OVERRIDES["hard_stop_frac"] = HARD_STOP_FRAC                          # hard per-trade stop (same in eval)
OVERRIDES["profit_hold_weight"] = PROFIT_HOLD_WEIGHT; OVERRIDES["profit_hold_min_bars"] = PROFIT_HOLD_MIN_BARS
OVERRIDES["fast_pass_bonus"] = FAST_PASS_BONUS; OVERRIDES["fast_pass_hours"] = FAST_PASS_HOURS
POLICY_NAME, _name_basis = auto_name(OVERRIDES)   # base_policy is supplied by RESUME_FROM in Barbershop

print(f"Episode: {EPISODE_DAYS} days/episode (one continuous account) | failed-day penalty={FAILED_DAY_PENALTY}")
print(f"Challenge: +{CHALLENGE.daily_target_pct}%/day, {CHALLENGE.daily_risk_pct}% wall | gamma/lambda LOCKED {GAMMA_LOCKED}/{LAMBDA_LOCKED}")
print(f"Reward weights: net_pnl={NET_PNL_WEIGHT} step_pnl={STEP_PNL_WEIGHT} daily_progress={DAILY_PROGRESS_WEIGHT} pain={DRAWDOWN_PAIN_WEIGHT} trade_quality={TRADE_QUALITY_WEIGHT}")
print(f"Aggression ranges: entropy={ENTROPY_RANGE} clip={CLIP_RANGE} lr={LR_RANGE} epochs={EPOCHS_RANGE} | floor(min_agg)={MIN_AGGRESSION} | rollout={ROLLOUT_SIZE}/{MINIBATCH_SIZE}")
print(f"OVERRIDES (diff vs baseline): {OVERRIDES if OVERRIDES else '{}  (pure baseline)'}")
print(f"Auto policy name: {POLICY_NAME}")

## 5. Train

This runs the **real PPO training loop** (milestone **M8** — `quantra/learning_system/trainer/trainer.py`):
each update **collects an on-policy rollout** → **GAE** (🔴 γ=0.997 / λ=0.97) → **K epochs of minibatch PPO**
(`ppo_loss`) → advances the **aggression scheduler** from the G8 missed-opportunity rate → **checkpoints** the brain.

Two cells below:
1. **Build the env** — loads your real MT5 bars via the data loader (Parquet cache → Drive → gdown-by-ID). If no
   bars are reachable (e.g. Drive not mounted), it falls back to a **clearly-labelled deterministic synthetic**
   series so the loop still runs as a smoke test. The env is built from the `CHALLENGE` / `REWARD_CFG` /
   `EPISODE_DAYS` objects from the HYPERPARAMETERS cell — nothing downstream is hardcoded.
2. **Train + checkpoint** — runs `Trainer.train(N_UPDATES)`, prints the per-update **training-health trace**
   (`approx_kl`, `clip_frac`, `value_loss`, `entropy`, `miss_rate`), proves the weights actually moved, and saves
   `artifacts/checkpoints/<policy>.pt` (tagged with `STATE_DIM` for the resume / compatibility gate) plus a
   `<policy>.history.json` trace.

> Raise `N_UPDATES` for a real run (50 is a quick proof). Watch `approx_kl` (should stay small / bounded) and
> `value_loss` (should stay finite) — those are the Risk Doctor's training-health signals. To **resume** a saved
> brain, load its `state_dict` into a fresh `PPOAgent` after the compatibility signature matches (see the
> Barbershop notebook Cell 4 for the gate).

In [ ]:
# ── 5a. Build the env the Trainer learns on (REAL bars; deterministic synthetic fallback) ──
# COUPLING -> quantra/market_pipeline/data_loader (load_symbol) + env/trading_env.py
#   (prepare_symbol_data, TradingEnv, SymbolData). The env consumes CHALLENGE / REWARD_CFG /
#   EPISODE_DAYS built in the HYPERPARAMETERS cell, so a knob change there changes what "passing"
#   means here — nothing is hardcoded downstream.
import numpy as np
from quantra.env.trading_env import TradingEnv, prepare_symbol_data, SymbolData

SYMBOL = "EURUSD"   # the binding case for real-bar training (see PROJECT_GUIDE §7)
try:
    from quantra.market_pipeline.data_loader import load_symbol
    df, rep = load_symbol(SYMBOL, path=None)                 # Parquet cache -> Drive -> gdown by ID
    SD = prepare_symbol_data(df, symbol=SYMBOL)              # builds the locked 207-feature obs (cached)
    usable = len(SD.close) - SD.valid_from
    if usable < 2000:
        raise RuntimeError(f"only {usable} warmed-up bars — too few to train")
    print(f"[data] REAL bars: {len(df):,}  warmup={SD.valid_from:,}  usable={usable:,}  source={rep.source}")
except Exception as e:
    # Deterministic synthetic fallback so the loop ALWAYS runs (smoke / verify). LOUDLY labelled, and
    # capped to a few days regardless of EPISODE_DAYS (synthetic is only for proving the loop turns).
    print(f"[data] real bars unavailable -> using DETERMINISTIC SYNTHETIC bars (smoke run only).\n        reason: {e}")
    from quantra.market_pipeline.feature_builder.schema import PRECOMPUTED_DIM, PRECOMPUTED_NAMES
    DAY = 1440; SYN_DAYS = min(max(EPISODE_DAYS, 4), 5); T = SYN_DAYS * DAY
    rng = np.random.default_rng(7)
    mat = np.zeros((T, PRECOMPUTED_DIM), dtype=np.float32)
    for nm, v in [("atr_dev_1m", .1), ("atr_dev_30m", .1), ("spread_range_ratio_1m", .3), ("adf_stat_1m", -3.5)]:
        mat[:, PRECOMPUTED_NAMES.index(nm)] = v             # open the 3 market-condition signals
    for nm in ("cci10_1m", "cci30_1m"):                      # a little signal for the policy to chew on
        if nm in PRECOMPUTED_NAMES:
            mat[:, PRECOMPUTED_NAMES.index(nm)] = rng.normal(0, 80, T)
    # Slow CCI "regime" so the TEMPORARY CCI-regime gate (if on) is actually exercised on the synthetic
    # smoke run: give CCI30/CCI100 on 1m+4H a slow wave + their SMAs the same sign at smaller scale, so
    # clean bull/bear stretches (and brief mixed zero-crossings) form. Real bars use real CCI/SMA.
    _reg = 120.0 * np.sin(np.arange(T) / (DAY / 2.0))
    for _p in (30, 100):
        _cci = _reg if _p == 30 else 0.9 * _reg
        for _tf in ("1m", "4H"):
            mat[:, PRECOMPUTED_NAMES.index(f"cci{_p}_{_tf}")]     = _cci
            mat[:, PRECOMPUTED_NAMES.index(f"cci{_p}_sma_{_tf}")] = 0.6 * _cci
    close = (1.20 * np.exp(np.cumsum(rng.normal(0, 4e-4, T)))).astype(float)
    SD = SymbolData(mat, close=close, atr=np.full(T, 1e-3), spread=np.full(T, 2e-5),
                    valid_from=0, dates=np.repeat(np.arange(T // DAY + 1), DAY)[:T])  # dates -> daily reset

# One env for the whole run; the account carries forward across EPISODE_DAYS (C10).
# cci_regime_gate=CCI_REGIME_GATE wires the TEMPORARY experiment into the TRAINING env too (off by
# default in the repo; the toggle lives in the HYPERPARAMETERS cell). COUPLING -> env/trading_env.py.
env = TradingEnv({SYMBOL: SD}, challenge=CHALLENGE, reward_cfg=REWARD_CFG, episode_days=EPISODE_DAYS,
                 cci_regime_gate=CCI_REGIME_GATE, risk_cfg=RISK_CFG)
print(f"[env] T={env.T:,} bars  episode_days={env.episode_days}  obs_width={len(env.reset())} (== STATE_DIM 207)"
      f"  CCI-regime gate={'ON' if env.cci_regime_gate else 'off'}  per-trade risk={env.risk_cfg.max_per_trade_risk_frac:.3%}")

In [ ]:
# ── 5b. THE PPO TRAINING LOOP — collect rollout -> GAE -> K-epoch minibatch PPO -> schedule -> checkpoint ──
# COUPLING -> quantra/learning_system/trainer/trainer.py (Trainer.train / .checkpoint),
#   ppo_agent/agent.py (PPOAgent reads STATE_DIM dynamically), trainer/scheduler.py
#   (AggressionScheduler picks dials within AGG_RANGES from the G8 miss-rate), barbershop_runner.run_pass
#   (the REAL per-day FTMO scoreboard used for the in-training day-by-day gauge below).
# WHY CHUNKED: train in REPORT_EVERY-update blocks so you watch progress LIVE (never wait blind on a long
#   run), and every EVAL_EVERY updates run a quick DETERMINISTIC pass that prints the per-DAY scoreboard
#   (day | PASS/miss/BREACH | pnl% | dd% | trades) — the literal "see the trainings day by day" view.
import json, torch
from pathlib import Path
from quantra.learning_system.trainer.trainer import Trainer
from quantra.learning_system.trainer.scheduler import AggressionScheduler
from quantra.learning_system.ppo_agent.agent import PPOAgent
from quantra.learning_system.barbershop_runner import run_pass
from quantra.market_pipeline.feature_builder import STATE_DIM

N_UPDATES       = 60000 # collect->update cycles. THIS is the long run (will NOT finish in one Colab CPU
                        #   session — periodic checkpoints below mean a disconnect isn't fatal; resume later).
REPORT_EVERY    = 250   # print a live training-health line every this many updates (so you're never blind)
EVAL_EVERY      = 2500  # every this many updates, run a quick deterministic pass + print the per-DAY scoreboard
CHECKPOINT_EVERY= 2500  # save the brain every this many updates so a long run survives a Colab disconnect
EVAL_DAYS_GAUGE = min(EVAL_DAYS, 8)   # how many days the in-training gauge scores (IN-SAMPLE progress gauge)

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
agent   = PPOAgent(device=DEVICE)                            # input dim read from schema (no hardcode)
trainer = Trainer(env, agent=agent, train_cfg=TRAIN_CFG,
                  scheduler=AggressionScheduler(AGG_RANGES, min_aggression=MIN_AGGRESSION), device=DEVICE)

_gate_on = bool(OVERRIDES.get("cci_regime_gate", False))
# Brand-new policy: random-init weights (NO checkpoint resume) + an EMPTY history. Nothing is carried
# over from any previous run — this is a clean start from scratch, exactly as requested.
assert len(trainer.history) == 0, "expected a FRESH Trainer with empty history (no resume)"
print(f"Training on {DEVICE} · {N_UPDATES} updates · rollout={TRAIN_CFG.rollout_size}/{TRAIN_CFG.minibatch}"
      f" · CCI-regime gate={'ON' if _gate_on else 'off'} · aggression floor={MIN_AGGRESSION}")
print("[fresh] brand-new policy (random init, no resume) · history empty\n")
_p0 = [p.detach().clone() for p in trainer.agent.net.parameters()]

def _print_health(done):
    h = trainer.history[-1]
    print(f"  upd {done:4d}/{N_UPDATES}: kl={h['approx_kl']:+.4f} clip={h['clip_frac']:.3f} "
          f"value_loss={h['value_loss']:.4f} ent={h.get('entropy', 0):.3f} "
          f"miss={h['miss_rate']:.3f} lr={h['lr']:.1e} agg={h['aggression']:.2f}")

def _print_day_scoreboard(done):
    # REAL per-day FTMO scoreboard for the CURRENT policy on the first EVAL_DAYS_GAUGE days (an IN-SAMPLE
    # progress gauge; the true OOS check is the Barbershop / walk-forward notebook). Honors OVERRIDES, so
    # the CCI-regime gate (if on) is enforced here exactly as in training. Soft-fails so a gauge hiccup
    # never kills the training run.
    try:
        rows = run_pass(trainer.agent, {SYMBOL: SD}, overrides=OVERRIDES, n_days=EVAL_DAYS_GAUGE)
    except Exception as e:
        print(f"  ── day-by-day gauge skipped after {done} updates ({type(e).__name__}: {e})\n")
        return
    n_pass = sum(r["passed"] for r in rows); n_breach = sum(r["breached"] for r in rows)
    _w = sum(r["wins"] for r in rows); _c = sum(r["closes"] for r in rows)
    _pool = (100.0 * _w / _c) if _c else 0.0     # pooled win rate (winning closes / closes) across the days
    print(f"  ── day-by-day after {done} updates (in-sample gauge, {len(rows)} days):")
    for r in rows:
        flag = "PASS  " if r["passed"] else ("BREACH" if r["breached"] else "miss  ")
        print(f"      day {r['day']:2d}: {flag}  pnl={r['pnl_pct']:+6.2f}%  dd={r['dd_pct']:+6.2f}%  trades={r['trades']:3d}  win={r['win_rate']:5.1f}% ({r['wins']}/{r['closes']} closes)")
    print(f"     -> {n_pass}/{len(rows)} passed · {n_breach} breached · win {_pool:.1f}% ({_w}/{_c} closes)\n")

# Chunked loop: train REPORT_EVERY updates, print a live health line, periodically print the day scoreboard.
done = 0
while done < N_UPDATES:
    chunk = min(REPORT_EVERY, N_UPDATES - done)
    trainer.train(chunk)            # continues training; appends to trainer.history
    done += chunk
    _print_health(done)
    if done % EVAL_EVERY == 0 or done == N_UPDATES:
        _print_day_scoreboard(done)
    if done % CHECKPOINT_EVERY == 0 and done < N_UPDATES:    # interim save -> a disconnect loses < CHECKPOINT_EVERY updates
        _ck = trainer.checkpoint(POLICY_NAME)
        Path(_ck).with_suffix('.history.json').write_text(json.dumps(trainer.history))
        print(f'   [ckpt] interim save @ {done} updates -> {_ck}')

# Proof a real gradient step happened, and the health trace stayed sane.
_delta  = sum((a - b).abs().sum().item() for a, b in zip(trainer.agent.net.parameters(), _p0))
_finite = all(np.isfinite(h['approx_kl']) and np.isfinite(h['value_loss']) for h in trainer.history)
print(f"\n[learn] weights moved Σ|Δ|={_delta:.1f}  (a real gradient step happened: {_delta > 0})")
print(f"[health] all approx_kl & value_loss finite across the run: {_finite}")

# Checkpoint the brain (tagged with STATE_DIM for the resume / compatibility gate) + the health trace.
CKPT = trainer.checkpoint(POLICY_NAME)                      # -> artifacts/checkpoints/<POLICY_NAME>.pt
_hp = Path(CKPT).with_suffix(".history.json"); _hp.write_text(json.dumps(trainer.history, indent=2))
print(f"[ckpt] saved {CKPT}  (state_dim={STATE_DIM})")
print(f"[hist] training-health trace -> {_hp}")
print("\nDone. Resume by loading this checkpoint's state_dict into a fresh PPOAgent once its compatibility")
print("signature matches (Barbershop Cell 4). Evaluate it with the Barbershop notebook (real per-day scoreboard).")